In [0]:
from delta.tables import *

stage_table_name = 'incremental_load_2.default.orders_data_stage'
target_table_name = 'incremental_load_2.default.orders_data_target'

In [0]:
spark.sql("SHOW TABLES IN incremental_load_2.default").show()


In [0]:
stage_df = spark.read.table(stage_table_name)

In [0]:
# create target table schema if target table doesn't exist

if not spark._jsparkSession.catalog().tableExists(target_table_name):
    stage_df.write.format('delta').saveAsTable(target_table_name)
else: 
    target_table = DeltaTable.forName(spark, target_table_name)

# Define merge condition

merge_condition = "stage.tracking_num =  target.tracking_num"

# Execute the merge operation
target_table.alias('target') \
    .merge(stage_df.alias('stage'), merge_condition) \
    .whenMatchedDelete() \
    .execute()

stage_df.write.format('delta').mode('append').saveAsTable(target_table_name)
